# MLP y Regresión logística

En este jupyternotebook se va a entrenar un MLP que pueda llegar a diferenciar la señal del fondo en dos casos. Uno con las variables solo PRI y otro con las variables PRI y DER. 

Para ello primero se va a desarrollar un modelo de regresión logística, se va a describir la función AMS que es la que va a permitir analizar si se ha logrado un buen resultado y por último el desarrollo MLP.

## Preparación de datos

Para desarrollar tanto la regresión logística como el MLP se va a hacer uso del csv de entrenamiento.

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [14]:
Entrenamiento = pd.read_csv("Entrenamiento.csv")
Ajuste = pd.read_csv("Ajuste.csv")
Test = pd.read_csv("Test.csv")

Este csv ya ha sometido a una normalización y limpieza de datos, por lo que solo es necesario ajustar los parametros para poder manipularlos en los diferentes modelos.

Los valores -999 no son valores que se tengan que eliminar como ya se ha observado en el SOM, por tanto se van a convertir en valores nulos para que no condicionen el resultado del MLP

In [15]:
Entrenamiento = Entrenamiento.replace(np.nan, 0)
Ajuste = Ajuste.replace(np.nan, 0)
Test = Test.replace(np.nan, 0)

Se va a intercambiar los valores de señal y fondo por 0 y 1

In [16]:
Entrenamiento['Label'] = Entrenamiento["Label"].replace({"s":1, "b":0}).astype(int)
Ajuste["Label"] = Ajuste["Label"].replace({"s":1, "b":0}).astype(int)
Test["Label"] = Test["Label"].replace({"s":1, "b":0}).astype(int)

y por último se generarán los dos subgrupos de PRI y PRI + DER junto con las columna del label que dice si es fondo o señal.

In [17]:
Entrenamiento_DER = Entrenamiento.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Entrenamiento_PRI = Entrenamiento_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)
L_E = Entrenamiento["Label"]

In [18]:
Ajuste_DER = Ajuste.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Ajuste_PRI = Ajuste_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)
L_A = Ajuste["Label"]
W_A = Ajuste["KaggleWeight"]

In [19]:
Test_DER = Test.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Test_PRI = Test_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)
L_T = Test["Label"]
W_T = Test["KaggleWeight"]

# Regresión logística

La regresión logística es modelo de aprendizaje supervisado lineal con una sola capa oculta. Mediante el desarrollo de la regresión logística se pretende comprobar si existe o no esa linealidad y por tanto si es necesario generar un MLP o si es sufiente con la regresión logística.

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [21]:
def calcular_ams(y, prob, pesos, umbral):                                  # el AMS permite ver cómo de acertado fue (cuanto más alto mejor)
    c = 10
    seleccionados = prob >= umbral                  # Genera un vector true false, siendo true si el evento de señal
    s = pesos[seleccionados & (y == 1)].sum()       # Suma la cantidad de eventos que sean señal tanto en el modelo como de verdad
    b = pesos[seleccionados & (y == 0)].sum()

    AMS = np.sqrt(2*((s+b+c)*np.log(1+(s/(b+c)))-s))
    return AMS

def buscar_mejor_umbral(y, prob, pesos):
    umbrales = np.quantile(prob, np.linspace(0.50, 0.99, 200))
    lista_ams = [calcular_ams(y, prob, pesos, u) for u in umbrales]
    mejor = np.argmax(lista_ams)
    return umbrales[mejor], lista_ams[mejor]

In [22]:
def regresion_logistica(Entrenamiento, L_E, Ajuste, L_A, W_A, Test, L_T, W_T, C=np.inf):
    # 1) Entrenar (pasos 1-3 del pseudocódigo)
    modelo = LogisticRegression(solver="newton-cholesky", C=C, tol=1e-7, max_iter=20)   # Crea el modelo
    modelo.fit(Entrenamiento, L_E)                  # Entrena hasta que deja de cambiar o max_iter

    print("Iteraciones:", modelo.n_iter_)           # cuántas vueltas dio el bucle
    print(modelo.intercept_, modelo.coef_)          # el vector β final: β₀ y β₁…β_d


    # 2) Elegir el umbral con el mejor AMS en VALIDACIÓN
    p_val = modelo.predict_proba(Ajuste)[:, 1]
    umbral, ams_val = buscar_mejor_umbral(L_A, p_val, W_A)

    # 3) Aplicar ese umbral al TEST
    p_test = modelo.predict_proba(Test)[:, 1]
    ams_test = calcular_ams(L_T, p_test, W_T, umbral)
    # auc_test = roc_auc_score(L_T, p_test, sample_weight = W_T)

    print(f"Umbral = {umbral:.3f} | AMS val = {ams_val:.3f} | AMS test = {ams_test:.3f}")
    return modelo, ams_test

In [23]:
modelo_pri, ams_pri = regresion_logistica(Entrenamiento_PRI, L_E, Ajuste_PRI, L_A, W_A, Test_PRI, L_T, W_T)
modelo_der, ams_der = regresion_logistica(Entrenamiento_DER, L_E, Ajuste_DER, L_A, W_A, Test_DER, L_T, W_T)

print(f"AMS PRI / AMS PRI+DER = {ams_pri / ams_der:.1%}")

Iteraciones: [4]
[-0.72550226] [[ 0.57123369 -0.00440606 -0.01019967 -0.21368881 -0.00151402  0.00378634
  -0.35406173  0.01067767  0.01582954 -0.27714944  0.34082926  0.00170435
   0.00075096 -0.23060109  0.00723793 -0.00837861  0.60880778]]
Umbral = 0.327 | AMS val = 1.641 | AMS test = 1.632
Iteraciones: [5]
[-0.84035351] [[ 2.10755372e-01 -4.49211244e-01 -8.00024037e-01  9.18948291e-02
   1.54167966e+00 -1.30730667e+00 -1.10542791e-01  8.84474950e-01
  -1.07950004e-01 -1.64589844e-01  7.82749190e+00  2.73320527e-01
   3.57774590e-01  7.04037723e+00 -2.94005488e-03 -5.93565855e-03
  -5.04756636e+00 -4.18131736e-03  5.41980232e-03 -1.65048770e-01
   3.18822105e-03  1.95898433e-02 -1.62139828e-01  4.53905813e-01
  -3.33523755e-04 -1.68504062e-04  3.01231922e-01  8.51095650e-03
  -4.28478207e-03  4.28696867e-01]]
Umbral = 0.385 | AMS val = 1.860 | AMS test = 1.879
AMS PRI / AMS PRI+DER = 86.9%


## Análisis de resultados